In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("/media/yannik/Intenso/DATA/dfg_plexus/radiomics_features___harmonized___combined___SA___normalized_ICV.csv", delimiter=";")

In [ ]:
df.columns

In [ ]:
site_df = df.copy()
is_hc = site_df["patID"].str.startswith("hc")
subject_num = site_df["patID"].str.extract(r'Subject(\d+)')[0].astype(float)
is_nic = is_hc | (subject_num > 588)
site_df["site"] = np.where(is_nic, 'NIC', 'NRAD')

In [ ]:
print(site_df['site'].value_counts())

In [ ]:
# 1. Define the 3 distinct groups
conditions = [
    site_df['patID'].str.startswith('hc'),                                  # Site A HC
    site_df['patID'].str.startswith('Subject') & (site_df['site'] == 'NIC'),     # Site A MS/CIS (The 3 samples)
    site_df['site'] == 'NRAD'                                               # Site B MS/CIS
]

choices = [
    'NIC - Healthy',
    'NIC - MS/CIS (N=3)',
    'NRAD - MS/CIS'
]

# Create the new grouping column
site_df['Cohort'] = np.select(conditions, choices, default='Unknown')

# 2. Print the mathematical means
print("--- Mean Values for original_firstorder_Maximum ---")
print(site_df.groupby('Cohort')['original_firstorder_Maximum'].mean())
print("-" * 50)

# 3. Plot the distributions
plt.figure(figsize=(10, 6))

# Boxplot for the overall distribution
sns.boxplot(
    data=site_df,
    x='Cohort',
    y='original_firstorder_Maximum',
    palette='Set2',
    showfliers=False # Hide standard outliers so the swarmplot handles them
)

# Swarmplot overlays the actual individual data points (vital for the N=3 group)
sns.swarmplot(
    data=site_df,
    x='Cohort',
    y='original_firstorder_Maximum',
    color='black',
    alpha=0.7,
    size=5
)

plt.title('Scanner Batch Effect Test: original_firstorder_Maximum')
plt.ylabel('Maximum Pixel Intensity')
plt.xlabel('')
plt.tight_layout()
plt.show()